In [16]:
import pandas as pd
import re

In [17]:
df = pd.read_csv('Reference/Data/Danh sách ủng hộ bão yagi - 01_09-10_09.csv')
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
df[['Amount','Unit']] = df['Amount'].str.split(' ', expand=True)

df = df[['Date', 'Amount', 'Unit', 'Note']]
df.head()

,Date,Amount,Unit,Note
0,2024-09-10,"1,000,000,000",₫,SHGD:10004067.DD:240910.BO:VAN PHONG HOC VIEN ...
1,2024-09-10,"500,000,000",₫,SHGD:10008948.DD:240910.BO:CTY TNHH OLALA MEDI...
2,2024-09-10,"400,000,000",₫,MBVCB.6997056113.Hoa Minzy cung FC xin duoc un...
3,2024-09-09,"300,000,000",₫,020097042209091712522024VHQZ187912.6995 8.1712...
4,2024-09-10,"300,000,000",₫,TAP THE LAI DO KDL TRANG AN UNG HO DONG BAO


In [18]:
# Extract person/organization name by RegEx
def extract_name (text):
    try:
        pattern = r'\b[A-ZÀ-Ỹ]{2,}(?:\s+[A-ZÀ-Ỹ]{2,}){1,5}\b'
        match = re.search(pattern, text.upper())
        if match:
            start, end = match.span()
            return text[start:end]
        return "Cannot be detected"
    except Exception:
        return "Can not be detected"
    
extract_name('MDFG342.75649.Tran thi ngoc chuyen tien CT tu 3529523583 den TW mat tran')

'Tran thi ngoc chuyen tien CT'

In [19]:
def clean_name(text):
    keywords = ['UNG HO', 'CHUYEN TIEN', 'CHUYEN KHOAN', 'DONG GOP', 'CHUNG TAY', 'GOP',\
                'QUYEN GOP', 'CUU TRO', 'HO TRO', 'CHIA SE', 'XIN', 'GIUP']
    pattern = r'(.+?)\s+(?:' + '|'.join(keywords) + r')\b'
    
    match = re.search(pattern, text, flags=re.IGNORECASE)
    if match:
        # If the keyword is at the start (no real name before), keep original
        if text.strip().upper().startswith(tuple(keywords)):
            return text
        else:
            return match.group(1).strip()
    return text

print(clean_name("tran thi anh hong gop chut suc luc chuyen tien ung ho dong bao lu lut"))
print(clean_name("1 chut suc luc"))

tran thi anh hong
1 chut suc luc


In [20]:
# Detect organization
org_patterns = [
    r'\bCTY\b',
    r'\bCTCP\b',
    r'\bCONG TY\b',
    r'\bTNHH\b',
    r'\bCHI NHANH\b',
    r'\bVAN PHONG\b',
    r'\bDOANH NGHIEP\b',
    r'\bJSC\b',
    r'\bTAP DOAN\b',
    r'\bGROUP\b',
    r'\bTEAM\b',
    r'\bTAP THE\b',
    r'\bDOI NGU\b',
    r'\bCLB\b',
    r'\bCAU LAC BO\b',
    r'\bFC\b',
    r'\bFAN\b',
    r'\bSHOP\b'
]

def is_org (text):
    text = text.upper()

    for pattern in org_patterns:
        if re.search(pattern, text):
            return True
    return False

In [21]:
df['Segment'] = df['Note'].apply(
    lambda x: "Organization" if is_org(x) else "Individual/Unknown"
)

In [22]:
df['Donator Name'] = df['Note'].apply(lambda x: extract_name(x))
df['Donator Name'] = df['Donator Name'].apply(lambda x: clean_name(x))

In [23]:
def clean_message(text):
    # remove any leading or trailing whitespace
    text = text.strip()
    # remove patterns like ".CT tu 3529523583 den TW mat tran" at the end of the text
    text = re.sub(r'\s*.CT\s+tu\b.*$', '', text, flags=re.IGNORECASE)
    # remove Vietcombank and other bank names
    text = re.sub(r'\b(VIETCOMBANK|AGRIBANK|BIDV|TECHCOMBANK|ACB|VPBANK|MB BANK|TPBANK|SHB|SCB|OCB|HDBANK|EXIMBANK|SEABANK|LIENVIETPOSTBANK)\b', '', text, flags=re.IGNORECASE)
    # Only keep text after pattern ".Remark:"
    text = re.split(r'\.Remark:', text, maxsplit=1)
    if len(text) > 1:
        text = text[1]
    else:
        text = text[0]
    # # remove any ID numbers and ID characters
    pattern = re.compile(
    r'^(?:[A-Za-zÀ-ỹ0-9._:/-]+(?:\s+[A-Za-zÀ-ỹ0-9._:/-]+)*\.)+'
    r'|(?:\s*(?:FT[0-9A-Za-zÀ-ỹ]+|[0-9]{4,}))\s*$'
    )
    text = pattern.sub('', text).strip()
    # remove date and time patterns
    text = re.sub(r'-\d+-\d+:\d+:\d+', '', text)
    return text

clean_message('689840.100924.183533.TIEM TRA AN THINH KENBAR UNG HO BA CON BAO SO 3-100924-18:34:31 689840')
# clean_message('SHGD:10006328.DD:240910.BO:CTY CP DAU TU VA XD THAI BINH DUONG.Remark:THAI BINH DUONG UNG HO DONG BAO LU LUT')
# clean_message('590130.100924.110709.Phuong HHL ung ho khac phuc hau qua bao so 3 Yagi')

'TIEM TRA AN THINH KENBAR UNG HO BA CON BAO SO 3'

In [24]:
df['Message'] = df['Note'].apply(lambda x: clean_message(x))

In [25]:
df = df.rename(columns={'Note': 'Trx Note'})
df = df[['Date', 'Amount', 'Unit', 'Trx Note', 'Message', 'Segment', 'Donator Name']]

In [26]:
df.head(5)

,Date,Amount,Unit,Trx Note,Message,Segment,Donator Name
0,2024-09-10,"1,000,000,000",₫,SHGD:10004067.DD:240910.BO:VAN PHONG HOC VIEN ...,HV CTQG HOCHIMINH UNG HO LE PHATDONG UNG HO DO...,Organization,VAN PHONG HOC VIEN CHINH TRI
1,2024-09-10,"500,000,000",₫,SHGD:10008948.DD:240910.BO:CTY TNHH OLALA MEDI...,UNG HO DONG BAO BI LU LUT VA ANH HUONG BAO YAGI,Organization,CTY TNHH OLALA MEDIA
2,2024-09-10,"400,000,000",₫,MBVCB.6997056113.Hoa Minzy cung FC xin duoc un...,Hoa Minzy cung FC xin duoc ung ho ba con bi th...,Organization,Hoa Minzy cung FC
3,2024-09-09,"300,000,000",₫,020097042209091712522024VHQZ187912.6995 8.1712...,Cong Dong Team Chau Phi ung ho khac phuc bao s...,Organization,Cong Dong Team Chau Phi ung
4,2024-09-10,"300,000,000",₫,TAP THE LAI DO KDL TRANG AN UNG HO DONG BAO,TAP THE LAI DO KDL TRANG AN UNG HO DONG BAO,Organization,TAP THE LAI DO KDL TRANG


In [ ]:
df.to_csv('Pre-processedData/Danh sách ủng hộ bão yagi - 01_09-10_09.csv', index=False)